# Assignment 2.1 — Variational Autoencoder (VAE) with Pure PyTorch

Dataset: **MNIST**

This notebook avoids `torchvision` completely to prevent Kaggle P100 dependency conflicts.

Components:
- Encoder: Conv layers → `mu`, `logvar`
- Reparameterization: `z = mu + eps * sigma`
- Decoder: ConvTranspose layers → reconstructed image
- Loss: BCE Reconstruction Loss + `beta * KL Divergence`


In [ ]:
# Cell 1: Install PyTorch version compatible with Tesla P100 (sm_60)
# Run before importing torch.

!pip uninstall -y -q torch torchvision torchaudio
!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126


In [ ]:
# Cell 2: Imports and GPU check
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Supported arch:", torch.cuda.get_arch_list())


In [ ]:
# Cell 3: Hyperparameters
batch_size = 128
latent_dim = 20
epochs = 10
learning_rate = 1e-3
beta = 1.0


In [ ]:
# Cell 4: Download and load MNIST without torchvision
mnist_path = "mnist.npz"

if not os.path.exists(mnist_path):
    url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    urllib.request.urlretrieve(url, mnist_path)

with np.load(mnist_path) as data:
    x_train = data["x_train"]
    y_train = data["y_train"]
    x_test = data["x_test"]
    y_test = data["y_test"]

# Convert [N, 28, 28] uint8 -> [N, 1, 28, 28] float32 in [0, 1]
x_train = torch.from_numpy(x_train).float().unsqueeze(1) / 255.0
x_test = torch.from_numpy(x_test).float().unsqueeze(1) / 255.0

y_train = torch.from_numpy(y_train).long()
y_test = torch.from_numpy(y_test).long()

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train samples:", len(train_dataset))
print("Test samples :", len(test_dataset))


In [ ]:
# Cell 5: Encoder
class Encoder(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # 28 -> 14
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 14 -> 7
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)

        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        return mu, logvar


In [ ]:
# Cell 6: Decoder
class Decoder(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 64 * 7 * 7)

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32, kernel_size=4, stride=2, padding=1
            ),  # 7 -> 14
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 1, kernel_size=4, stride=2, padding=1
            ),  # 14 -> 28
            nn.Sigmoid()
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), 64, 7, 7)
        return self.deconv(x)


In [ ]:
# Cell 7: Complete VAE
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()

        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)

        return x_recon, mu, logvar


In [ ]:
# Cell 8: VAE Loss
def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction="sum"
    )

    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss


In [ ]:
# Cell 9: Model and optimizer
model = VAE(latent_dim=latent_dim).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print(model)


In [ ]:
# Cell 10: Train
train_losses = []

for epoch in range(1, epochs + 1):
    model.train()

    total_loss = 0
    total_recon = 0
    total_kl = 0

    for x, _ in train_loader:
        x = x.to(device)

        optimizer.zero_grad()

        x_recon, mu, logvar = model(x)

        loss, recon_loss, kl_loss = vae_loss(
            x_recon,
            x,
            mu,
            logvar,
            beta
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    avg_loss = total_loss / len(train_dataset)
    avg_recon = total_recon / len(train_dataset)
    avg_kl = total_kl / len(train_dataset)

    train_losses.append(avg_loss)

    print(
        f"Epoch [{epoch:02d}/{epochs}] "
        f"Loss: {avg_loss:.4f} | "
        f"Recon: {avg_recon:.4f} | "
        f"KL: {avg_kl:.4f}"
    )


In [ ]:
# Cell 11: Plot training loss
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss per sample")
plt.title("VAE Training Loss")
plt.grid(True)
plt.show()


In [ ]:
# Cell 12: Original vs reconstructed images
model.eval()

x, _ = next(iter(test_loader))
x = x[:10].to(device)

with torch.no_grad():
    x_recon, _, _ = model(x)

x = x.cpu()
x_recon = x_recon.cpu()

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")

    axes[1, i].imshow(x_recon[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Recon")

plt.tight_layout()
plt.show()


In [ ]:
# Cell 13: Generate new samples
model.eval()

with torch.no_grad():
    z = torch.randn(16, latent_dim, device=device)
    generated = model.decoder(z).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))

for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Generated MNIST Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 14: Save model
torch.save(model.state_dict(), "vae_mnist.pth")
print("Saved: vae_mnist.pth")
